<a href="https://colab.research.google.com/github/Riya541/Machine-learning-internship/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('neo (3).csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nHead:\n", df.head())
print("\nData info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget column value counts:")
# check potential target columns
for col in df.columns:
    if 'hazard' in col.lower() or 'is_p' in col.lower() or 'target' in col.lower():
        print(f"\nValue counts for {col}:")
        print(df[col].value_counts(dropna=False))

Shape: (90836, 10)

Columns: ['id', 'name', 'est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'orbiting_body', 'sentry_object', 'absolute_magnitude', 'hazardous']

Head:
         id                 name  est_diameter_min  est_diameter_max  \
0  2162635  162635 (2000 SS164)          1.198271          2.679415   
1  2277475    277475 (2005 WK4)          0.265800          0.594347   
2  2512244   512244 (2015 YE18)          0.722030          1.614507   
3  3596030          (2012 BV13)          0.096506          0.215794   
4  3667127          (2014 GE35)          0.255009          0.570217   

   relative_velocity  miss_distance orbiting_body  sentry_object  \
0       13569.249224   5.483974e+07         Earth          False   
1       73588.726663   6.143813e+07         Earth          False   
2      114258.692129   4.979872e+07         Earth          False   
3       24764.303138   2.543497e+07         Earth          False   
4       42737.733765   4.627557e+0

In [ ]:
print("orbiting_body unique:", df['orbiting_body'].nunique(), df['orbiting_body'].unique())
print("sentry_object unique:", df['sentry_object'].nunique(), df['sentry_object'].unique())

orbiting_body unique: 1 ['Earth']
sentry_object unique: 1 [False]


In [ ]:
df['ratio'] = df['est_diameter_max'] / df['est_diameter_min']
print("Ratio min/max statistics:\n", df['ratio'].describe())

Ratio min/max statistics:
 count    9.083600e+04
mean     2.236068e+00
std      5.101698e-09
min      2.236068e+00
25%      2.236068e+00
50%      2.236068e+00
75%      2.236068e+00
max      2.236068e+00
Name: ratio, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

# Clean dataset copy
data = df.copy()

# Feature Engineering
data['est_diameter_avg'] = (data['est_diameter_min'] + data['est_diameter_max']) / 2.0
# Drop constant & id columns
data_clean = data.drop(columns=['id', 'name', 'orbiting_body', 'sentry_object', 'est_diameter_min', 'est_diameter_max', 'ratio'])

# Additional features
data_clean['log_miss_distance'] = np.log1p(data_clean['miss_distance'])
data_clean['log_relative_velocity'] = np.log1p(data_clean['relative_velocity'])
data_clean['log_diameter_avg'] = np.log1p(data_clean['est_diameter_avg'])
data_clean['kinetic_energy_proxy'] = 0.5 * (data_clean['est_diameter_avg']**3) * (data_clean['relative_velocity']**2)
data_clean['close_approach_risk'] = data_clean['relative_velocity'] / (data_clean['miss_distance'] + 1e-5)

X = data_clean.drop(columns=['hazardous'])
y = data_clean['hazardous'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training shapes:", X_train.shape, "Testing shapes:", X_test.shape)

# LightGBM model with scale_pos_weight
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
lgb_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, scale_pos_weight=scale_pos, verbose=-1)
lgb_model.fit(X_train, y_train)

y_pred = lgb_model.predict(X_test)
y_prob = lgb_model.predict_proba(X_test)[:, 1]

print("\n--- LightGBM Classification Report ---")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

precision, recall, _ = precision_recall_curve(y_test, y_prob)
print("PR-AUC:", auc(recall, precision))

Training shapes: (72668, 9) Testing shapes: (18168, 9)

--- LightGBM Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.78      0.88     16400
           1       0.33      0.97      0.49      1768

    accuracy                           0.80     18168
   macro avg       0.66      0.88      0.68     18168
weighted avg       0.93      0.80      0.84     18168

ROC-AUC: 0.924318335448626
PR-AUC: 0.551655255361715


In [ ]:
for pkg in ['lightgbm', 'sklearn', 'seaborn', 'matplotlib', 'joblib', 'gradio', 'streamlit', 'imblearn']:
    try:
        __import__(pkg)
        print(f"{pkg}: available")
    except ImportError:
        print(f"{pkg}: NOT available")

lightgbm: available
sklearn: available
seaborn: available
matplotlib: available
joblib: available
gradio: available
streamlit: NOT available
imblearn: available


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, confusion_matrix

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"=== {name} ===")
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    print("-" * 50)

=== Logistic Regression ===
ROC-AUC: 0.5388 | PR-AUC: 0.1078
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000     16400
           1     0.0973    1.0000    0.1774      1768

    accuracy                         0.0973     18168
   macro avg     0.0487    0.5000    0.0887     18168
weighted avg     0.0095    0.0973    0.0173     18168

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


=== Random Forest ===
ROC-AUC: 0.9268 | PR-AUC: 0.5659
              precision    recall  f1-score   support

           0     0.9309    0.9780    0.9539     16400
           1     0.6151    0.3264    0.4265      1768

    accuracy                         0.9146     18168
   macro avg     0.7730    0.6522    0.6902     18168
weighted avg     0.9002    0.9146    0.9025     18168

--------------------------------------------------
=== HistGradientBoosting ===
ROC-AUC: 0.9212 | PR-AUC: 0.5429
              precision    recall  f1-score   support

           0     0.9970    0.7785    0.8743     16400
           1     0.3226    0.9785    0.4852      1768

    accuracy                         0.7979     18168
   macro avg     0.6598    0.8785    0.6798     18168
weighted avg     0.9314    0.7979    0.8364     18168

--------------------------------------------------
